In [1]:
import xarray as xr
import dask
import numpy as np
import fsspec
import zarr
import time
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from xclim.indices import standardized_precipitation_index
from google.oauth2 import service_account


In [11]:
import xarray as xr
import dask
import numpy as np
import time
import zarr
import gcsfs
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar
from xclim.indices import standardized_precipitation_index
from google.oauth2 import service_account

def apply_spi_chrips(cont_db):
    """
    Calculates the 3-month Standardized Precipitation Index (SPI) from an input dataset.
    """
    cont_db['precip'].attrs['units'] = 'mm/month'
    
    # Extract precipitation data
    aa = cont_db.precip

    # Compute SPI-3 using xclim (Dask-aware)
    spi_3 = standardized_precipitation_index(
        aa,
        freq="MS",         
        window=3,         
        dist="gamma",       
        method="APP",      
        cal_start='1991-01-01',
        cal_end='2018-01-01',
        fitkwargs={"floc": 0}
    )

    # Create a proper dataset with coordinates
    spi_dataset = xr.Dataset(
        {"spi3": spi_3},
        coords=cont_db.coords
    )
    
    # Add metadata
    spi_dataset.spi3.attrs.update({
        'long_name': 'Standardized Precipitation Index (3-month)',
        'units': 'unitless',
        'description': 'SPI calculated using gamma distribution with APP method'
    })
    
    # Return without computing
    return spi_dataset

def test_spi_calculation_local():
    """Testing function for local SPI calculation with small subset."""
    start_time = time.time()
    
    print("Setting up local Dask cluster...")
    # Create a local Dask cluster with resources appropriate for n2-standard-2
    # n2-standard-2 has 2 vCPUs and 8GB memory
    cluster = LocalCluster(
        n_workers=2,           # Use 2 workers for 2 vCPUs
        threads_per_worker=1,  # 1 thread per worker to avoid oversubscription
        memory_limit="3GiB",   # Limit memory to avoid OOM (leaving some for system)
    )
    
    client = Client(cluster)
    print(f"Dask dashboard: {client.dashboard_link}")
    
    try:
        # Path to service account credentials
        credentials_path = 'coiled-data-e4drr.json'
        
        # Define the required scopes for read/write access
        scopes = ["https://www.googleapis.com/auth/devstorage.read_write"]
        
        # Create credentials object with the required scope
        credentials = service_account.Credentials.from_service_account_file(
            credentials_path, scopes=scopes
        )
        
        # Create GCS filesystem with the credentials
        fs = gcsfs.GCSFileSystem(token=credentials)
        
        # Input CHIRPS Zarr path in GCS (assuming this is a Zarr v3 store)
        zarr_path = "gs://ittseas51/ea_chirps_v20_monthly_20250415.zarr"
        
        # Storage options for xarray to use with GCS
        storage_options = {'token': credentials}
        
        # Output path for test results (local for testing)
        output_zarr_path = "./test_spi3_output.zarr"
        
        # Open the dataset properly for Zarr v3
        print("Opening CHIRPS dataset...")
        ds = xr.open_dataset(
            zarr_path, 
            engine='zarr', 
            consolidated=False,
            storage_options=storage_options
        )
        
        print("Original dataset info:")
        print(ds)
        
        # Create a very small subset for testing
        # Small spatial area (just a few grid cells) and limited time period
        print("Creating small test subset...")
        test_subset = ds.sel(
            time=slice('2010-01-01', '2020-12-31'),   # Limit to recent decade
            latitude=slice(0, 2),                     # Just 2 degrees (few grid cells)
            longitude=slice(35, 37)                   # Just 2 degrees (few grid cells)
        )
        
        # Set chunks appropriate for small test
        test_subset = test_subset.chunk({
            'time': -1,          # No chunking in time (needed for SPI)
            'latitude': 1,       # Small chunks for testing
            'longitude': 1
        })
        
        print("Test subset info:")
        print(test_subset)
        print(f"Test subset chunking: {test_subset.chunks}")
        
        # Calculate memory size (approximate)
        nbytes = test_subset.nbytes
        print(f"Approximate memory size of test subset: {nbytes / 1e6:.2f} MB")
        
        # Persist the small subset to memory
        print("Persisting test subset...")
        with ProgressBar():
            test_subset = client.persist(test_subset)
            # Wait for completion
            test_subset = test_subset.compute()
        
        # Calculate SPI-3
        print("Calculating SPI-3 on test subset...")
        spi_result = apply_spi_chrips(test_subset)
        
                # Check the actual shape of the data
        print("Shape of SPI result:", spi_result.spi3.shape)
        print("Dimensions:", spi_result.spi3.dims)
        
        # Get dimension lengths
        time_len = spi_result.spi3.sizes['time']
        lat_len = spi_result.spi3.sizes['latitude']
        lon_len = spi_result.spi3.sizes['longitude']
        
        # Define chunk sizes based on data dimensions
        # Ensure chunks are always smaller than or equal to dimension size
        output_chunks = {
            'time': min(12, time_len),             # Time in yearly chunks (or less)
            'latitude': min(10, lat_len),          # Latitude in reasonable chunks
            'longitude': min(10, lon_len)          # Longitude in reasonable chunks
        }
        
        print("Using chunk sizes:", output_chunks)
        
        # Create a proper chunking tuple that matches the data shape and dimension order
        dims = spi_result.spi3.dims  # Get the dimensions in correct order
        chunks_tuple = tuple(output_chunks[dim] for dim in dims)
        
        print("Chunk tuple:", chunks_tuple)
        
        # Now create the encoding with the correct chunks format
        encoding = {
            'spi3': {
                'compressor': None,
                'dtype': 'float32',
                '_FillValue': -9999.0,
                'chunks': chunks_tuple
            }
        }
        
        # For the to_zarr call, use the chunking dictionary with the Xarray method
        spi_result = spi_result.chunk(output_chunks)
        
        # Then do your write operation
        with ProgressBar():
            write_job = spi_result.to_zarr(
                output_zarr_path,
                mode='w',
                encoding=encoding,
                consolidated=True,
                compute=False
            )
            
            # Execute the write task
            future = client.compute(write_job)
            
            # Wait for completion
            future.result()
        
        # Verify the output
        print("Verifying output...")
        result_ds = xr.open_zarr(output_zarr_path)
        print(result_ds)
        
        # Calculate simple statistics to verify data
        with ProgressBar():
            spi_min = float(result_ds.spi3.min().compute())
            spi_max = float(result_ds.spi3.max().compute())
            spi_mean = float(result_ds.spi3.mean().compute())
        
        print(f"SPI-3 stats: min={spi_min:.3f}, max={spi_max:.3f}, mean={spi_mean:.3f}")
        
        # If you want to write back to GCS, use the same approach:
        print("Writing test output to GCS (optional)...")
        output_gcs_path = "gs://ittseas51/test_spi3_output.zarr"
        
        with ProgressBar():
            # Use to_dataset for Zarr v3 format
            write_gcs_job = spi_result.to_zarr(
                output_gcs_path,
                mode='w',
                encoding=encoding,
                consolidated=True,
                storage_options=storage_options,
                compute=False
            )
            
            # Execute the GCS write
            future_gcs = client.compute(write_gcs_job)
            future_gcs.result()
        
        total_time = time.time() - start_time
        print(f"Test completed successfully in {total_time:.2f} seconds.")
        
    except Exception as e:
        print(f"Error during test: {str(e)}")
        raise
    
    finally:
        # Clean up
        client.close()
        cluster.close()
        print("Local cluster closed.")

if __name__ == "__main__":
    test_spi_calculation_local()

Setting up local Dask cluster...
Dask dashboard: https://cluster-rktlq.dask.host/jupyter/proxy/8787/status
Opening CHIRPS dataset...
Original dataset info:
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 531, latitude: 1000, longitude: 1000)
Coordinates:
  * latitude   (latitude) float32 4kB -19.98 -19.93 -19.88 ... 29.88 29.92 29.97
  * time       (time) datetime64[ns] 4kB 1981-01-01 1981-02-01 ... 2025-03-01
  * longitude  (longitude) float32 4kB 10.02 10.07 10.12 ... 59.88 59.93 59.97
Data variables:
    precip     (time, latitude, longitude) float32 2GB ...
Attributes: (12/15)
    Conventions:       CF-1.6
    title:             CHIRPS Version 2.0
    history:           created by Climate Hazards Group
    version:           Version 2.0
    date_created:      2025-04-14
    creator_name:      Pete Peterson
    ...                ...
    reference:         Funk, C.C., Peterson, P.J., Landsfeld, M.F., Pedreros,...
    comments:           time variable denotes the first day of the gi

/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/core/group.py:2469: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Verifying output...
<xarray.Dataset> Size: 3MB
Dimensions:       (time: 132, latitude: 40, longitude: 40)
Coordinates:
    prob_of_zero  (time, latitude, longitude) float64 2MB dask.array<chunksize=(12, 10, 10), meta=np.ndarray>
  * latitude      (latitude) float32 160B 0.025 0.075 0.125 ... 1.925 1.975
  * longitude     (longitude) float32 160B 35.02 35.07 35.12 ... 36.93 36.97
  * time          (time) datetime64[ns] 1kB 2010-01-01 2010-02-01 ... 2020-12-01
Data variables:
    spi3          (time, latitude, longitude) float32 845kB dask.array<chunksize=(12, 10, 10), meta=np.ndarray>
SPI-3 stats: min=-5.764, max=5.943, mean=0.208
Writing test output to GCS (optional)...


/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/core/group.py:2469: UserWarning: The `compressor` argument is deprecated. Use `compressors` instead.
  compressors = _parse_deprecated_compressor(
/opt/conda/envs/itt/lib/python3.11/site-packages/zarr/api/asynchronous.py:203: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Test completed successfully in 83.87 seconds.
Local cluster closed.
